# Strands Agents with Bedrock AgentCore Code Interpreter — FSI Edition

This lab demonstrates how to use Amazon Bedrock AgentCore Code Interpreter to give your AI agent the ability to execute Python code dynamically — applied to financial services use cases.

## Overview

In this lab, you will:
- Use the default Code Interpreter to run financial calculations in a sandbox
- Analyze transaction data for fraud patterns
- Calculate portfolio risk metrics (VaR, sector concentration)
- Create a custom Code Interpreter with network access for live market data

## Why Code Interpreter for FSI?

Financial services require:
- **Dynamic calculations** — Risk models, stress tests, scenario analysis
- **Data analysis** — Fraud detection, anomaly identification
- **Secure execution** — Sandboxed environment for sensitive financial data
- **Audit trail** — Every calculation is traceable

## Prerequisites

Ensure you have AWS credentials configured and Nova Pro model access enabled.

In [ ]:
import os

#os.environ["AWS_ACCESS_KEY_ID"] = ""
#os.environ["AWS_SECRET_ACCESS_KEY"] = ""
#os.environ["AWS_SESSION_TOKEN"] = ""
#os.environ["AWS_REGION"] = ""

In [ ]:
#%pip install -q strands-agents strands-agents-tools rich bedrock-agentcore pandas

In [1]:
import boto3

region = boto3.session.Session().region_name

NOVA_PRO_MODEL_ID = "us.amazon.nova-pro-v1:0"
if region.startswith("eu"):
    NOVA_PRO_MODEL_ID = "eu.amazon.nova-pro-v1:0"
elif region.startswith("ap"):
    NOVA_PRO_MODEL_ID = "apac.amazon.nova-pro-v1:0"

print(f"Region: {region}")
print(f"Nova Pro Model ID: {NOVA_PRO_MODEL_ID}")

Region: ap-southeast-2
Nova Pro Model ID: apac.amazon.nova-pro-v1:0


## Part 1: Default Code Interpreter — Financial Calculations

The default Code Interpreter runs Python in a **sandboxed environment** with no network access. Perfect for secure financial calculations.

Let's test it with a portfolio risk calculation:

In [2]:
from strands import Agent
from strands.models import BedrockModel
from strands_tools.code_interpreter import AgentCoreCodeInterpreter

# Initialize the AgentCore Code Interpreter (default: sandboxed, no network)
agentcore_code_interpreter = AgentCoreCodeInterpreter()

# Create agent with default Code Interpreter (sandboxed)
risk_agent = Agent(
    model=BedrockModel(model_id=NOVA_PRO_MODEL_ID),
    system_prompt="""You are a quantitative analyst assistant. You write and execute Python code 
    to perform financial calculations. Always show the code you execute and explain the results 
    in plain language.""",
    tools=[agentcore_code_interpreter.code_interpreter],
)

risk_agent("""Calculate the monthly compound interest on a $2,000,000 investment 
at 4.8% annual rate over 5 years. Show me a year-by-year breakdown of the balance.""")

tool_name=<<module 'strands_tools.code_interpreter' from '/Users/zohaibso/AI Workshops/FSI-AgentCore-Workshop/.venv/lib/python3.11/site-packages/strands_tools/code_interpreter/__init__.py'>> | failed to load tool
Traceback (most recent call last):
  File "/Users/zohaibso/AI Workshops/FSI-AgentCore-Workshop/.venv/lib/python3.11/site-packages/strands/tools/registry.py", line 117, in add_tool
    tools = load_tools_from_module(tool, module_tool_name)
            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/zohaibso/AI Workshops/FSI-AgentCore-Workshop/.venv/lib/python3.11/site-packages/strands/tools/loader.py", line 136, in load_tools_from_module
    raise AttributeError(
AttributeError: The module code_interpreter is not a valid module for loading tools.This module must contain @tool decorated function(s), or must be a module based tool.


ValueError: Failed to load tool <module 'strands_tools.code_interpreter' from '/Users/zohaibso/AI Workshops/FSI-AgentCore-Workshop/.venv/lib/python3.11/site-packages/strands_tools/code_interpreter/__init__.py'>: The module code_interpreter is not a valid module for loading tools.This module must contain @tool decorated function(s), or must be a module based tool.

## Part 2: Fraud Detection on Transaction Data

Now let's give the agent our synthetic transaction dataset and ask it to identify fraud patterns.

The dataset (`data/transactions.csv`) contains 25 transactions with several suspicious patterns:
- **Velocity attack** — Multiple high-value transactions within seconds
- **Geo-anomaly** — Transactions in different countries within minutes
- **Escalating amounts** — Progressively larger transactions (testing limits)
- **Unusual timing** — High-value transactions at 3am

In [ ]:
# Load the transaction data so we can pass it to the agent
import pandas as pd

transactions_df = pd.read_csv("../data/transactions.csv")
print(f"Loaded {len(transactions_df)} transactions")
transactions_df.head()

In [ ]:
# Pass the data as context and ask the agent to analyze it
transaction_data = transactions_df.to_csv(index=False)

risk_agent(f"""Analyze the following transaction data for fraud patterns. 
Look for:
1. Velocity attacks (multiple transactions in rapid succession)
2. Geographic anomalies (impossible travel - transactions in different countries within minutes)
3. Amount anomalies (unusual amounts or escalating patterns)
4. Timing anomalies (unusual hours for high-value transactions)

Write Python code to detect these patterns and flag suspicious transactions.
Provide a risk score (1-10) for each flagged customer.

Transaction data:
{transaction_data}""")

## Part 3: Portfolio Risk Analysis (VaR)

Let's analyze a portfolio using Value at Risk (VaR) — a standard risk metric in financial services.

We'll use the portfolio data from `data/portfolio.csv`.

In [ ]:
portfolio_df = pd.read_csv("../data/portfolio.csv")
print(f"Loaded {len(portfolio_df)} positions")
portfolio_df.head(10)

In [ ]:
portfolio_data = portfolio_df.to_csv(index=False)

risk_agent(f"""Analyze this portfolio for risk metrics. Calculate:
1. Total portfolio value (current prices × units) for each client
2. Sector concentration — what % is in each sector? Flag if any sector > 30%
3. Unrealized P&L per position (current vs purchase price)
4. Asset class allocation (Equity vs Fixed Income vs Cash vs Other)

Present results as a clear summary with any risk warnings.

Portfolio data:
{portfolio_data}""")

## Part 4: Custom Code Interpreter with Network Access

The default Code Interpreter is sandboxed (no internet). For use cases that need live data (e.g., fetching real stock prices), we create a **custom Code Interpreter with network access**.

### Step 1: Initialize AgentCore Clients

In [ ]:
from bedrock_agentcore.runtime import AgentCoreApp
from bedrock_agentcore.services.code_interpreter import CodeInterpreterService

# Initialize the Code Interpreter service
ci_service = CodeInterpreterService()

print("✅ AgentCore Code Interpreter service initialized")

### Step 2: Create Custom Code Interpreter with Network Access

In [ ]:
# Create a custom code interpreter with public network access
custom_ci = ci_service.create_code_interpreter(
    name="fsi-risk-analyzer",
    network_access="PUBLIC"
)

print(f"✅ Custom Code Interpreter created: {custom_ci.code_interpreter_id}")
print(f"   Network access: PUBLIC (can fetch live market data)")

### Step 3: Create a Session and Test Live Data Access

In [ ]:
# Create a session in the custom code interpreter
session = custom_ci.create_session()

print(f"✅ Session created: {session.session_id}")

# Test: install yfinance and fetch a real stock price
result = session.execute_code("""
import subprocess
subprocess.run(['pip', 'install', '-q', 'yfinance'], capture_output=True)

import yfinance as yf
cba = yf.Ticker('CBA.AX')
info = cba.info
print(f"CBA.AX Live Price: ${info.get('currentPrice', 'N/A')} AUD")
print(f"Market Cap: ${info.get('marketCap', 0)/1e9:.1f}B AUD")
print(f"P/E Ratio: {info.get('trailingPE', 'N/A')}")
""")

print(result.output)

### Step 4: Use Custom Code Interpreter with Strands Agent

In [ ]:
from strands import Agent, tool
from strands.models import BedrockModel

@tool
def execute_python(code: str) -> str:
    """Execute Python code in a secure sandbox with internet access.
    Can install packages with pip and fetch live data.
    
    Args:
        code: Python code to execute
    """
    result = session.execute_code(code)
    return result.output if result.output else "Code executed successfully (no output)"

# Create agent with custom code interpreter
live_agent = Agent(
    model=BedrockModel(model_id=NOVA_PRO_MODEL_ID),
    system_prompt="""You are a quantitative analyst with access to live market data.
    You can execute Python code to fetch real-time prices, calculate risk metrics,
    and generate analysis. Use yfinance for market data.""",
    tools=[execute_python],
)

live_agent("Fetch the current prices of the big 4 Australian banks (CBA, WBC, NAB, ANZ) and compare their P/E ratios.")

## Examining the Agent Loop

In [ ]:
from rich.table import Table
import rich
import json

console = rich.get_console()

console.print("Agent Loop Detail")
console.rule()
console.print(f"Number of Loops: {live_agent.event_loop_metrics.cycle_count}")

table = Table(title="Agent Messages", show_lines=True)
table.add_column("Role", style="green")
table.add_column("Text", style="magenta", max_width=60)
table.add_column("Tool Name", style="cyan")
table.add_column("Tool Input", style="cyan", max_width=40)
table.add_column("Tool Result", style="cyan", max_width=40)

for message in live_agent.messages:
    text = [content["text"] for content in message["content"] if "text" in content]
    tool_name = [content["toolUse"]["name"] for content in message["content"] if "toolUse" in content]
    tool_input = [content["toolUse"]["input"] for content in message["content"] if "toolUse" in content]
    tool_result = [content["toolResult"]["content"][0] for content in message["content"] if "toolResult" in content]
    table.add_row(
        message["role"], (text[-1][:200] + "...") if text and len(text[-1]) > 200 else (text[-1] if text else ""),
        tool_name[-1] if tool_name else "",
        (json.dumps(tool_input[-1])[:150] + "...") if tool_input else "",
        (json.dumps(tool_result[-1])[:150] + "...") if tool_result else ""
    )

console.print(table)

## Resource Cleanup (Optional)

Clean up the custom Code Interpreter to avoid charges:

In [ ]:
# Uncomment to clean up
# session.close()
# custom_ci.delete()
# print("✅ Resources cleaned up")

## Summary

In this lab, you:

- ✅ Used the default Code Interpreter for secure financial calculations
- ✅ Analyzed transaction data for fraud patterns (velocity, geo-anomaly, timing)
- ✅ Calculated portfolio risk metrics (sector concentration, P&L, allocation)
- ✅ Created a custom Code Interpreter with network access for live market data
- ✅ Fetched real-time stock prices and compared bank P/E ratios

### FSI Takeaways

| Capability | FSI Application |
|-----------|----------------|
| Sandboxed execution | Secure risk calculations on sensitive data |
| Dynamic code generation | Ad-hoc analysis without pre-built reports |
| Network-enabled interpreter | Live market data, API integrations |
| Audit trail (agent loop) | Compliance — every calculation is traceable |

### Next: Lab 02 — Browser Automation
We'll use AgentCore Browser to monitor regulatory websites (APRA, ASX) and extract live financial data.